In [1]:
import pandas as pd
import math

# charger le dataset diabetes
data = pd.read_csv("../data/diabetes.csv")

# la colonne target (0 ou 1)
target_col = data.columns[-1]
# toutes les colonnes sauf target (features)
features = list(data.columns[:-1])


# séparer les données par classe
def separate_by_class(data):
    # dictionnaire pour stocker les classes
    separated = {}
    # on parcourt toutes les lignes du dataset
    for i in range(len(data)):
        row = data.iloc[i]          # une ligne (un patient)
        label = row[target_col]     # la classe (0 ou 1)
        # si la classe n'existe pas encore dans le dictionnaire
        if label not in separated:
            separated[label] = []
        # on ajoute la ligne dans sa classe
        separated[label].append(row)

    return separated

#  moyenne
def mean(values):
    # calcul simple de la moyenne
    return sum(values) / len(values)

# variance
def variance(values):
    avg = mean(values)
    # mesure de la dispersion des données
    return sum((x - avg) ** 2 for x in values) / len(values)


# résumé des données
# ici on calcule les statistiques (moyenne et variance)
# pour chaque classe (0 et 1) et pour chaque feature
def summarize_by_class(separated):

    # dictionnaire final qui va contenir tous les résultats
    summaries = {}

    # on parcourt chaque classe du dataset
    # label = 0 ou 1
    # rows = toutes les lignes de cette classe
    for label, rows in separated.items():

        # on crée un sous-dictionnaire pour cette classe
        summaries[label] = {}
        # on parcourt toutes les colonnes (features)
        for feature in features:
            # on récupère toutes les valeurs de cette colonne
            # pour cette classe uniquement
            values = [row[feature] for row in rows]
            # on calcule la moyenne de ces valeurs
            # ça donne une idée générale de la feature
            mean_value = mean(values)

            # on calcule la variance
            # ça mesure à quel point les valeurs sont dispersées
            var_value = variance(values)

            # on stocke les résultats dans le dictionnaire
            summaries[label][feature] = {
                "mean": mean_value,   # valeur moyenne
                "var": var_value      # dispersion
            }

    # on retourne tout le résumé
    return summaries
    
# fonction gaussienne
def gaussian(x, mean, var):

    # si la variance est 0
    # ça veut dire que toutes les valeurs sont identiques
    # donc on retourne 1 pour éviter la division par zéro
    if var == 0:
        return 1

    # partie principale de la formule gaussienne
    # elle calcule à quel point x est proche de la moyenne
    # plus x est proche de mean plus la valeur est grande
    exponent = math.exp(-((x - mean) ** 2) / (2 * var))

    # formule complète de la distribution normale
    # elle normalise la valeur pour obtenir une probabilité
    return (1 / math.sqrt(2 * math.pi * var)) * exponent

#  prédiction pour une ligne
def predict_row(summaries, row):
    probabilities = {}

    # on calcule la probabilité pour chaque classe
    for label in summaries:
        # on commence avec 1 (multiplication des probabilités)
        probabilities[label] = 1
        # on parcourt chaque feature
        for feature in features:
            mean = summaries[label][feature]["mean"]
            var = summaries[label][feature]["var"]
            x = row[feature]
            # on multiplie les probabilités (hypothèse naïve)
            probabilities[label] *= gaussian(x, mean, var)

    # on choisit la classe avec la plus grande probabilité
    return max(probabilities, key=probabilities.get)


#  entraînement
def train_naive_bayes(data):

    # séparer les données par classe
    separated = separate_by_class(data)
    # calculer statistiques (mean + variance)
    summaries = summarize_by_class(separated)

    return summaries


# test du modèle
model = train_naive_bayes(data)

correct = 0

# on teste sur tout le dataset
for i in range(len(data)):

    row = data.iloc[i]

    # prédiction du modèle
    pred = predict_row(model, row)

    # comparaison avec la vraie valeur
    if pred == row[target_col]:
        correct += 1

# calcul de l’accuracy
accuracy = correct / len(data)

print("Accuracy:", accuracy)

Accuracy: 0.7513020833333334
